# 4-3 Series 绘图

Pandas 的 `Series.plot()` 可以直接使用 Series 中的数据绘图，底层默认使用 Matplotlib。
折线图通常使用索引作为横轴、数值作为纵轴。

本节先理解累加，再通过 `kind` 选择不同图表。

> 约定：使用 `np` 作为 NumPy 的别名，使用 `plt` 作为 `matplotlib.pyplot` 的别名。
> 下方准备单元格会选择可用的中文字体；可以从项目根目录或本章目录运行。
> `fonts` 文件夹是可选的，没有该文件夹时会查找系统字体。
> 如果没有可用的中文字体，会显示提示并使用默认字体继续绘图，中文可能显示为方框。
> 字体设置只作用于当前 Python 进程，详细原理见第 4-6 节。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# fonts 文件夹是可选的；没有字体文件时，使用系统已安装的字体。
project_dir = Path('.')
if Path.cwd().name == '第4章-Matplotlib':
    project_dir = Path('..')
font_files = [
    project_dir / 'fonts' / 'NotoSansCJKsc-Regular.otf',
    project_dir / 'fonts' / 'NotoSansSC-Regular.ttf',
]
font_path = next((path for path in font_files if path.is_file()), None)

if font_path is None:
    font_candidates = [
        'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
        'Source Han Sans SC', 'PingFang SC', 'Heiti SC', 'Songti SC',
        'Arial Unicode MS',
    ]
    available_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    font_name = next((name for name in font_candidates if name in available_fonts), None)
    if font_name is None:
        font_name = 'DejaVu Sans'
        print('未找到中文字体，暂用 DejaVu Sans，图中的中文可能显示为方框。')
        print('可安装 Noto Sans CJK SC，或将中文字体文件放入项目的 fonts 文件夹后重新运行。')
    font_path = Path(font_manager.findfont(
        font_manager.FontProperties(family=font_name), fallback_to_default=False
    ))
else:
    font_name = font_manager.FontProperties(fname=str(font_path)).get_name()

font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd

## 1. 使用 `cumsum()` 进行累加

`cumsum()` 返回累计和：结果中的每个位置，都是原数据从开头到该位置的总和。

In [ ]:
s = pd.Series(np.arange(5), name='原数据')
pd.DataFrame({'原数据': s, '累计和': s.cumsum()})

原数据 `[0, 1, 2, 3, 4]` 的累计和为 `[0, 1, 3, 6, 10]`。
`cumsum()` 返回新结果，原来的 `s` 不变。

## 2. 准备绘图数据

对 500 个随机数累加，形成一条上下波动的序列。固定随机种子可以使图形复现。

In [ ]:
rng = np.random.default_rng(42)
s1 = pd.Series(rng.standard_normal(500), name='累计值').cumsum()
s1.head()

## 3. 通过 `kind` 选择图表类型

| `kind` | 图表类型 | 主要用途 |
|---|---|---|
| `'line'` | 折线图，默认类型 | 观察有序数据的趋势 |
| `'hist'` | 直方图 | 观察数值分布 |
| `'bar'` | 竖向条形图 | 比较少量类别 |
| `'barh'` | 横向条形图 | 类别名称较长时便于阅读 |
| `'box'` | 箱线图 | 查看中位数、分散程度与潜在异常值 |
| `'kde'` | 核密度估计曲线 | 平滑展示分布形状 |

`grid` 控制网格，`label` 设置标签；图例通常还需要 `legend=True` 或 `plt.legend()`。

### 3.1 折线图：索引与数值

先使用 Matplotlib 明确传入索引和值。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(s1.index, s1.to_numpy(), label='累计值')
plt.xlabel('样本序号')
plt.ylabel('累计值')
plt.title('通过 Matplotlib 绘制 Series')
plt.legend()
plt.tight_layout()
plt.show()

使用 `Series.plot()` 可以得到相同的数据曲线。它返回一个 Matplotlib `Axes` 对象，
可以继续用 `set_xlabel()`、`set_ylabel()` 等方法设置图表。

In [ ]:
ax = s1.plot(kind='line', figsize=(7, 4), grid=True,
             label='累计值', legend=True, title='通过 Series.plot() 绘制折线')
ax.set_xlabel('样本序号')
ax.set_ylabel('累计值')
plt.tight_layout()
plt.show()

### 3.2 直方图：观察数值分布

此时横轴是数值区间，不再是 Series 的索引；纵轴表示每个区间包含多少个样本。

In [ ]:
ax = s1.plot(kind='hist', bins=15, rwidth=0.7, figsize=(7, 4), title='累计值的分布')
ax.set_xlabel('累计值')
ax.set_ylabel('频数')
plt.tight_layout()
plt.show()

### 3.3 竖向条形图

类别太多时图表容易拥挤，因此只取前 20 个值。使用 `.iloc[:20]` 明确按位置切片。

In [ ]:
first_20 = s1.iloc[:20]
ax = first_20.plot(kind='bar', figsize=(8, 4), rot=0, title='前 20 个累计值')
ax.set_xlabel('样本序号')
ax.set_ylabel('累计值')
plt.tight_layout()
plt.show()

### 3.4 横向条形图

`kind='barh'` 把类别放在纵轴、数值放在横轴。条形朝右表示正值，朝左表示负值。

In [ ]:
ax = first_20.plot(kind='barh', figsize=(7, 6), title='前 20 个累计值：横向条形图')
ax.set_xlabel('累计值')
ax.set_ylabel('样本序号')
plt.tight_layout()
plt.show()

### 3.5 箱线图

- 箱体的下边界和上边界分别是第 1 四分位数 `Q1` 和第 3 四分位数 `Q3`。
- 箱体中的横线表示中位数，四分位距为 `IQR = Q3 - Q1`。
- 默认须线延伸到距箱体不超过 `1.5 × IQR` 范围内的最远样本。
- 须线之外的点表示潜在异常值，不一定是错误数据。

In [ ]:
ax = s1.plot(kind='box', figsize=(5, 4), title='累计值的箱线图')
ax.set_xlabel('数据序列')
ax.set_ylabel('累计值')
plt.tight_layout()
plt.show()

### 3.6 KDE：核密度估计曲线

KDE 用一条平滑曲线估计数值分布。横轴是数据值，纵轴是概率密度，曲线下面积为 `1`。
密度曲线不是样本随时间变化的折线。

> Pandas 绘制 `kind='kde'` 需要可导入的 SciPy。数据也应包含足够多的、并非全部相同的数值。

先观察累计值的密度曲线。

In [ ]:
ax = s1.plot(kind='kde', figsize=(7, 4), title='累计值的核密度估计')
ax.set_xlabel('累计值')
ax.set_ylabel('概率密度')
plt.tight_layout()
plt.show()

再生成一组未经累加的标准正态样本。它的密度曲线通常接近以 `0` 为中心的钟形，
而累加序列的分布不应直接当作标准正态分布。

In [ ]:
s2 = pd.Series(rng.standard_normal(500), name='标准正态样本')
ax = s2.plot(kind='kde', figsize=(7, 4), title='标准正态样本的核密度估计')
ax.set_xlabel('样本值')
ax.set_ylabel('概率密度')
plt.tight_layout()
plt.show()

## 4. 要点总结

- `cumsum()` 计算累计和，适合生成累计变化序列。
- `Series.plot()` 默认绘制折线，以索引为横轴、数值为纵轴。
- `kind` 可以切换折线图、直方图、条形图、箱线图和 KDE。
- 绘图返回 `Axes`，可以继续设置标题、坐标轴说明和图例。
- 直方图、箱线图和 KDE 关注分布，不保留数据的先后变化趋势。
- KDE 使用概率密度，并需要 SciPy 支持。